In [2]:
import json
import os

with open('labeled_nouns.json', 'r', encoding='utf-8') as f:
    labeled_nouns_all = {
        record['data']['text']: [c['value']['choices'][0] for c in record['annotations'][0]['result']]
        for record in json.loads(f.read())
    }
    labeled_nouns = {key: labeled_nouns_all[key] for key in labeled_nouns_all if '杂项' not in labeled_nouns_all[key]}

len(labeled_nouns_all), len(labeled_nouns)

(2631, 1894)

In [2]:
set([item for sublist in list(labeled_nouns.values()) for item in sublist])

{'事故风险',
 '交通要素与流动',
 '交通设施',
 '人员活动',
 '功能设计',
 '安全设施与防护措施',
 '心理与感知因素',
 '情感状态',
 '感知评估',
 '治安管理',
 '活动与行为动态',
 '活动类型',
 '流量与密度',
 '物理基础设施',
 '环境暗示',
 '环境状况与维护',
 '监控与警示',
 '社会与治安因素',
 '社会秩序',
 '管理措施',
 '自然与天气因素',
 '行为与秩序',
 '行为模式',
 '视觉设计',
 '设计与美学要素',
 '防护设备',
 '隐患位置',
 '风险与隐患点'}

In [7]:
type_label = {
    "事故风险": "Accident Risk",
    "交通要素与流动": "Traffic Elements and Flow",
    "交通设施": "Transportation Infrastructure",
    "人员活动": "Human Activities",
    "功能设计": "Functional Design",
    "安全设施与防护措施": "Safety Facilities and Protective Measures",
    "心理与感知因素": "Psychological and Perceptual Factors",
    "情感状态": "Emotional State",
    "感知评估": "Perceptual Assessment",
    "治安管理": "Public Security Management",
    "活动与行为动态": "Activity and Behavioral Dynamics",
    "活动类型": "Types of Activities",
    "流量与密度": "Flow and Density",
    "物理基础设施": "Physical Infrastructure",
    "环境暗示": "Environmental Cues",
    "环境状况与维护": "Environmental Conditions and Maintenance",
    "监控与警示": "Surveillance and Warning Systems",
    "社会与治安因素": "Social and Public Safety Factors",
    "社会秩序": "Social Order",
    "管理措施": "Management Measures",
    "自然与天气因素": "Natural and Weather Factors",
    "行为与秩序": "Behavior and Order",
    "行为模式": "Behavioral Patterns",
    "视觉设计": "Visual Design",
    "设计与美学要素": "Design and Aesthetic Elements",
    "防护设备": "Protective Equipment",
    "隐患位置": "Hazard Locations",
    "风险与隐患点": "Risk and Hazard Hotspots"
}

sent_label = {
    "褒义": "Positive",
    "贬义": "Negative",
    "中性": "Neutral",
    "中型": "Neutral"
}

In [8]:
def generate_datarow(item, role, round):
    global labeled_nouns, type_label, sent_label
    
    response_file = f"Results/response_{item}_{role}_{round}.json"
    parse_file = f"Results/parse_{item}_{role}_{round}.json"

    if not os.path.exists(response_file):
        return None
    if not os.path.exists(parse_file):
        return None

    with open(response_file, 'r', encoding='utf-8') as f:
        response_json = json.loads(json.load(f)["output"]["choices"][0]["message"]["content"][0]["text"].replace("```json", "").replace("```", ""))
        score = response_json['score']
    
    datarow = {
        'item': item,
        'role': role,
        'round': round,
        'score': score
    }

    with open(parse_file, 'r', encoding='utf-8') as f:
        binary_pairs = json.loads(json.loads(f.read())["output"]["text"])
        if 'binary_pairs' in binary_pairs:
            binary_pairs = binary_pairs['binary_pairs']
        for pair in binary_pairs:
            noun = pair['noun'] if 'noun' in pair else pair['名词']
            sent = pair['sentiment'] if 'sentiment' in pair else pair['情感']
            if noun in labeled_nouns:
                for type in labeled_nouns[noun]:
                    varname = f'{type_label[type]} ({sent_label[sent]})'
                    if varname in datarow:
                        datarow[varname] += 1
                    else:
                        datarow[varname] = 1

    return datarow

generate_datarow(1, 'male', 2)

{'item': 1,
 'role': 'male',
 'round': 2,
 'score': 75,
 'Environmental Conditions and Maintenance (Positive)': 1,
 'Transportation Infrastructure (Positive)': 2,
 'Accident Risk (Negative)': 2,
 'Design and Aesthetic Elements (Neutral)': 1,
 'Flow and Density (Negative)': 1,
 'Emotional State (Negative)': 3,
 'Risk and Hazard Hotspots (Negative)': 2,
 'Physical Infrastructure (Negative)': 2,
 'Functional Design (Negative)': 1,
 'Psychological and Perceptual Factors (Neutral)': 1}

In [ ]:
data_root = "Data/StreetViews"
positions = [(item, os.path.join(data_root, item).replace("\\", "/")) for item in os.listdir(data_root)]

datarows = []
for position in positions:
    item, _ = position[0], position[1]
    for role in ["female", "male", "local", "non-local", "experience", "non-experience"]:
        for round in range(1, 3):
            file = f"Results/parse_{item}_{role}_{round}.json"
            datarow = generate_datarow(item, role, round)
            if datarow != None:
                datarows.append(datarow)

In [10]:
import pandas
pandas.DataFrame(datarows).fillna(0).to_excel('explain.xlsx', index=False)

C:\Users\PiesPro\AppData\Local\Temp\ipykernel_38636\1071610235.py:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.8' currently installed).
  pandas.DataFrame(datarows).fillna(0).to_excel('explain.xlsx', index=False)
